# 02 — Exploratory Data Analysis

Commercial questions answered with interactive Plotly figures.

**Questions covered**
- How does revenue move through the year?
- When do customers buy (hour / weekday)?
- How concentrated is product and customer revenue?
- Which countries and products matter most?
- How large is the returns problem?


In [11]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

pd.set_option("display.float_format", "{:.2f}".format)

reg = pd.read_csv(ROOT / "data" / "registered_clean.csv", parse_dates=["InvoiceDate"])
guests = pd.read_csv(ROOT / "data" / "guests.csv", parse_dates=["InvoiceDate"])
cancels = pd.read_csv(ROOT / "data" / "cancellations.csv", parse_dates=["InvoiceDate"])

print(f"Registered: {len(reg):,} lines | Guests: {len(guests):,} | Cancels: {len(cancels):,}")

Registered: 397,884 lines | Guests: 132,220 | Cancels: 9,288


C:\Users\ahmed\AppData\Local\Temp\ipykernel_6060\827797585.py:16: DtypeWarning: Columns (0: InvoiceNo) have mixed types. Specify dtype option on import or set low_memory=False.
  guests = pd.read_csv(ROOT / "data" / "guests.csv", parse_dates=["InvoiceDate"])


## Monthly revenue & the Q4 surge

In [12]:
monthly = (
    reg.groupby(reg["InvoiceDate"].dt.to_period("M"))
    .agg(Revenue=("LineRevenue", "sum"), Orders=("InvoiceNo", "nunique"), Customers=("CustomerID", "nunique"))
    .reset_index()
)
monthly["Month"] = monthly["InvoiceDate"].astype(str)

fig = px.area(monthly, x="Month", y="Revenue", title="Monthly Revenue (registered customers)")
fig.update_traces(line_color="#3b82f6", fillcolor="rgba(59,130,246,0.18)")
fig.update_layout(template="plotly_white", yaxis_tickprefix="£", height=420)
fig.show()

monthly[["Month", "Revenue", "Orders", "Customers"]]

,Month,Revenue,Orders,Customers
0,2010-12,572713.89,1400,885
1,2011-01,569445.04,987,741
2,2011-02,447137.35,997,758
3,2011-03,595500.76,1321,974
4,2011-04,469200.36,1149,856
5,2011-05,678594.56,1555,1056
6,2011-06,661213.69,1393,991
7,2011-07,600091.01,1331,949
8,2011-08,645343.90,1280,935
9,2011-09,952838.38,1755,1266


## Hourly traffic

In [13]:
hourly = reg.groupby("Hour").agg(Orders=("InvoiceNo", "nunique"), Revenue=("LineRevenue", "sum")).reset_index()
fig = px.bar(hourly, x="Hour", y="Orders", title="Orders by Hour of Day")
fig.update_traces(marker_color="#10b981")
fig.update_layout(template="plotly_white", height=380)
fig.show()

## Day-of-week pattern

In [14]:
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow = (
    reg.groupby("DayOfWeek")
    .agg(Revenue=("LineRevenue", "sum"), Orders=("InvoiceNo", "nunique"))
    .reindex(dow_order)
    .reset_index()
)
fig = px.bar(dow, x="DayOfWeek", y="Revenue", title="Revenue by Weekday")
fig.update_traces(marker_color="#6366f1")
fig.update_layout(template="plotly_white", yaxis_tickprefix="£", height=380)
fig.show()

## Average Order Value & basket size

In [15]:
order_level = (
    reg.groupby("InvoiceNo")
    .agg(
        Revenue=("LineRevenue", "sum"),
        Items=("Quantity", "sum"),
        Lines=("StockCode", "count"),
        CustomerID=("CustomerID", "first"),
    )
    .reset_index()
)
print("Order-level summary")
order_level[["Revenue", "Items", "Lines"]].describe()

Order-level summary


,Revenue,Items,Lines
count,18532.00,18532.00,18532.00
mean,480.87,278.86,21.47
std,1678.20,972.72,24.92
min,0.38,1.00,1.00
25%,158.04,74.00,6.00
50%,303.04,155.00,15.00
75%,471.84,291.00,28.00
max,168469.60,80995.00,542.00


## Pareto — product concentration

In [16]:
prod = (
    reg.groupby(["StockCode", "Description"])
    .agg(Revenue=("LineRevenue", "sum"), Qty=("Quantity", "sum"))
    .sort_values("Revenue", ascending=False)
    .reset_index()
)
prod["CumPct"] = prod["Revenue"].cumsum() / prod["Revenue"].sum() * 100
n80 = (prod["CumPct"] <= 80).sum()
print(f"{n80} products ({n80/len(prod)*100:.1f}% of catalogue) generate 80% of revenue")

fig = px.line(prod.head(300), x=prod.head(300).index + 1, y="CumPct",
              labels={"x": "Product rank", "CumPct": "Cumulative revenue %"},
              title="Pareto curve – product concentration")
fig.add_hline(y=80, line_dash="dash", line_color="#ef4444", annotation_text="80%")
fig.update_layout(template="plotly_white", height=400)
fig.show()

809 products (20.8% of catalogue) generate 80% of revenue


## Geographic performance

In [17]:
geo = (
    reg.groupby("Country")
    .agg(Revenue=("LineRevenue", "sum"), Customers=("CustomerID", "nunique"), Orders=("InvoiceNo", "nunique"))
    .assign(AOV=lambda x: x["Revenue"] / x["Orders"])
    .sort_values("Revenue", ascending=False)
    .reset_index()
)
geo.head(12)

,Country,Revenue,Customers,Orders,AOV
0,United Kingdom,7308391.55,3920,16646,439.05
1,Netherlands,285446.34,9,94,3036.66
2,EIRE,265545.90,3,260,1021.33
3,Germany,228867.14,94,457,500.80
4,France,209024.05,87,389,537.34
5,Australia,138521.31,9,57,2430.20
6,Spain,61577.11,30,90,684.19
7,Switzerland,56443.95,21,51,1106.74
8,Belgium,41196.34,25,98,420.37
9,Sweden,38378.33,8,36,1066.06


## Cancellations & returns

In [18]:
print(f"Cancellation lines: {len(cancels):,}")
print(f"Cancellation revenue (absolute): £{cancels['LineRevenue'].abs().sum():,.0f}")

top_return_skus = (
    cancels.groupby(["StockCode", "Description"])
    .agg(ReturnQty=("Quantity", "sum"), ReturnValue=("LineRevenue", "sum"))
    .assign(ReturnQty=lambda x: x["ReturnQty"].abs(), ReturnValue=lambda x: x["ReturnValue"].abs())
    .sort_values("ReturnValue", ascending=False)
    .head(15)
)
top_return_skus

Cancellation lines: 9,288
Cancellation revenue (absolute): £896,812


,,ReturnQty,ReturnValue
StockCode,Description,,
AMAZONFEE,AMAZON FEE,32,235281.59
23843,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.60
M,Manual,4066,146784.46
23166,MEDIUM CERAMIC TOP STORAGE JAR,74494,77479.64
POST,POSTAGE,147,11871.24
22423,REGENCY CAKESTAND 3 TIER,857,9722.55
CRUK,CRUK Commission,16,7933.43
BANK CHARGES,Bank Charges,25,7340.64
85123A,WHITE HANGING HEART T-LIGHT HOLDER,2578,6624.30


## Guest vs registered behaviour

In [19]:
print("Registered AOV:", reg.groupby("InvoiceNo")["LineRevenue"].sum().mean())
print("Guest AOV:     ", guests.groupby("InvoiceNo")["LineRevenue"].sum().mean())

Registered AOV: 480.8659563997409
Guest AOV:      1227.466181818182


## Customer-level concentration

In [20]:
cust_rev = reg.groupby("CustomerID")["LineRevenue"].sum().sort_values(ascending=False)
cum = cust_rev.cumsum() / cust_rev.sum() * 100
print(f"Top 10% of customers generate {cum.iloc[int(len(cum)*0.1)]:.1f}% of revenue")
print(f"Top 20% of customers generate {cum.iloc[int(len(cum)*0.2)]:.1f}% of revenue")

Top 10% of customers generate 61.4% of revenue
Top 20% of customers generate 74.6% of revenue
